<a href="https://colab.research.google.com/github/ttn64681/Bird-Audio-Classification/blob/main/hubert_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install required libraries ──────────────────────────────────────
!pip install -q transformers datasets evaluate torch pandas matplotlib seaborn scikit-learn kagglehub librosa fvcore

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import torch
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa

from transformers import (
    AutoFeatureExtractor, # was AutoTokenizer
    AutoModelForAudioClassification, # was AutoModelForSequenceClassification
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
)
import evaluate

# Used for preprocessing the audio datasets
from torch.utils.data import DataLoader, random_split
from sklearn.preprocessing import LabelEncoder

# Print PyTorch and device information
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Load HuBERT model ───────────────────────────────────────────────

# Use Facebook's HuBERT base model
MODEL_CHECKPOINT = "facebook/hubert-base-ls960"

# Use AutoFeatureExtractor instead of AutoTokenizer (for compatibility with audio, not text)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
print(f"Feature extractor loaded: {MODEL_CHECKPOINT}")

In [ ]:
# Get access to my Google Drive (for loading the datasets)
from google.colab import drive
drive.mount('/content/drive')

import os
# Check what is actually inside that folder
print(os.listdir('/content/drive/MyDrive/HuBERT/BirdCLEF/'))

In [ ]:
!cp -r /content/drive/MyDrive/HuBERT/BirdCLEF/ /content/BirdCLEF_local/

In [ ]:
import os
import librosa
import numpy as np
from datasets import Dataset, DatasetDict

def audio_generator(data_path, limit_per_class=500):
    # Get sorted folder names (bird species)
    species = sorted([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])
    label2id = {name: i for i, name in enumerate(species)}

    for bird in species:
        print(f"Extracting official features for: {bird}")
        folder_path = os.path.join(data_path, bird)
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.ogg', '.wav', '.mp3'))]

        count = 0
        for f in files:
            if count >= limit_per_class: break

            file_path = os.path.join(folder_path, f)
            try:
                # 1. Load the audio
                audio, _ = librosa.load(file_path, sr=16000)

                # 2. OFFICIAL HUGGING FACE PREPROCESSING
                encoded = feature_extractor(
                    audio,
                    sampling_rate=16000,
                    max_length=160000,
                    truncation=True,
                    padding="max_length",
                    return_tensors="np"
                )

                # 3. Yield to the dataset builder
                yield {
                    "input_values": encoded["input_values"][0],
                    "label": label2id[bird]
                }
                count += 1
            except Exception as e:
                print(f"Skipping {f} due to error: {e}")

# ── RUN THE GENERATOR ──────────────────────────────────────────────────
print("Building dataset file-by-file (This will prevent RAM crashes)...")
hf_dataset = Dataset.from_generator(
    audio_generator,
    gen_kwargs={"data_path": '/content/drive/MyDrive/HuBERT/BirdCLEF_local/'}
)

# ── SPLIT THE DATASET (70/20/10) ──────────────────────────────────────
print("Splitting into Train, Val, and Test...")
first_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
second_split = first_split['train'].train_test_split(test_size=0.222, seed=42)

final_datasets = DatasetDict({
    'train': second_split['train'],
    'validation': second_split['test'],
    'test': first_split['test']
})

# --- SAVE TO DRIVE ---
save_path = '/content/drive/MyDrive/HuBERT_Thai/bird_hf_datasets_official'
final_datasets.save_to_disk(save_path)
print(f"Success! Official dataset saved to: {save_path}")

In [ ]:
# Rebuild the label dictionaries globally
species = sorted([d for d in os.listdir('/content/drive/MyDrive/HuBERT/BirdCLEF/') if os.path.isdir(os.path.join('/content/drive/MyDrive/HuBERT/BirdCLEF/', d))])
labels = species
label2id = {name: i for i, name in enumerate(species)}
id2label = {i: name for i, name in enumerate(species)}
print(f"Unique labels: {labels}")

In [ ]:
from datasets import load_from_disk

# ── Load datasets from preprocessed files ─────────────────────────────────────────────────
def load_datasets(path):
    print(f'Loading HF Datasets from {path}/bird_hf_datasets...')
    loaded_datasets = load_from_disk(f'{path}/bird_hf_datasets_official') # <----------- change if need

    train_dataset = loaded_datasets['train']
    val_dataset = loaded_datasets['validation']
    test_dataset = loaded_datasets['test']

    print("Datasets have been loaded.")
    return train_dataset, val_dataset, test_dataset

# ── Metric functions for measuring accuracy ─────────────────────────────────────────────────

# Load evaluate metrics
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    # Accuracy metric
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    prec = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    rec = recall_metric.compute(predictions=predictions, references=labels, average="macro")
    # Return a dict with both scores
    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"],
        "precision": prec["precision"],
        "recall": rec["recall"]
    }


In [ ]:
# ── Load Datasets ─────────────────────────────────────────────────

# 1. Load the Master Datasets (the full 500-per-bird set)
train_dataset, val_dataset, test_dataset = load_datasets('/content/drive/MyDrive/HuBERT_Thai/')

# 2. Optionally slice each one to maintain the 70/20/10 ratio for 150-per-bird set
# We shuffle first to make sure we get a mix of all 10 birds in each slice
train_dataset = train_dataset.shuffle(seed=42).select(range(1050)) # 70% scaled
val_dataset   = val_dataset.shuffle(seed=42).select(range(300))    # 20% scaled
test_dataset  = test_dataset.shuffle(seed=42).select(range(150))   # 10% scaled

print(f"--- Scaled Dataset Sizes (70/20/10) ---")
print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

In [ ]:
# ── Evaluate Base HuBERT (Untrained) ─────────────────────────────────────────────────

print("=" * 50)
print("Evaluating Base HuBERT (Untrained Classification Head)")
print("=" * 50)

# 1. Load a fresh, untrained model
base_model = AutoModelForAudioClassification.from_pretrained(
    "facebook/hubert-base-ls960",
    num_labels=10
)

# 2. Create a temporary trainer just for evaluation
base_trainer = Trainer(
    model=base_model,
    args=TrainingArguments(output_dir="/tmp/base_eval", per_device_eval_batch_size=16, report_to="none"),
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# 3. Predict and print
base_results = base_trainer.predict(test_dataset)
print(f"  Base Accuracy   : {base_results.metrics['test_accuracy']:.4f}")
print(f"  Base F1 Score   : {base_results.metrics['test_f1']:.4f}")



In [ ]:
# --- FREE UP GPU MEMORY ---
import gc

# Delete the base model and trainer from memory so the GPU has room to train
del base_model
del base_trainer
gc.collect()
torch.cuda.empty_cache()
print("Cleared GPU memory for fine-tuning!")

In [ ]:
# ── Setup to Fine-tune HuBERT ─────────────────────────────────────────────────

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_CHECKPOINT, # Load the pretrained checkpoint (HuBERT-base)
    num_labels=10, # 10 bird species
)

training_args = TrainingArguments(
    output_dir="/content/training_checkpoints",
    num_train_epochs=10, # Train for 8 epochs
    learning_rate=2e-5, # very small learning rate (0.00002). Makes sense for fine tuning
    per_device_train_batch_size=8, # Training batch size is 16
    per_device_eval_batch_size=16, # Evaluation batch size is 32
    seed=42, # Using random seed 42
    eval_strategy="epoch", # Evaluate and save after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True, # After training, reload the best performing version of the model for evaluation
    metric_for_best_model="accuracy", # Metric used to decide which checkpoint is the best (uses accuracy function from part 4)
    save_total_limit=1,
    report_to="none",
)

trainer_birdclef = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# ── Train HuBERT ─────────────────────────────────────────────────

trainer_birdclef.train()

In [ ]:
# After training finishes:
print("Saving the BEST model permanently to Google Drive...")
# This only saves the final weights (~300MB), not the 10 epochs of junk
trainer_birdclef.save_model("/content/drive/MyDrive/HuBERT_Thai/bert_model_150_10_8_16")
print("Done!")

In [ ]:
trainer_birdclef.compute_metrics = compute_metrics

# ── Report final validation accuracy and F1 ───────────────────────

# Parameter Count (Model Size)
num_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {num_params / 1e6:.2f} Million")

# Training Speed (Samples per second)
# We pull this from the trainer state after training is done
if len(trainer_birdclef.state.log_history) > 0:
    train_metrics = trainer_birdclef.state.log_history[-1]
    # Sometimes the last entry is just an eval log, so we check for the key
    if 'train_samples_per_second' in train_metrics:
        print(f"Training throughput: {train_metrics['train_samples_per_second']:.2f} samples/sec")

print("-" * 50)

# Inference Latency + Final Accuracy (Calculated together)
import time
start_inference = time.time()

# predict() does exactly what evaluate() does, but gives us the raw predictions too
results = trainer_birdclef.predict(test_dataset)

end_inference = time.time()

# Calculate Latency logic
total_time = end_inference - start_inference
latency_per_sample = (total_time / len(test_dataset)) * 1000

# Print Everything
print("=" * 50)
print("BirdCLEF Fine-Tuned — Final Test Results (Test Set)")
print("=" * 50)
# Note: Results from predict() are stored in results.metrics
print(f"  Accuracy   : {results.metrics['test_accuracy']:.4f}")
print(f"  F1 Score   : {results.metrics['test_f1']:.4f}")
print(f"  Precision  : {results.metrics['test_precision']:.4f}")
print(f"  Recall     : {results.metrics['test_recall']:.4f}")
print("-" * 50)
print(f"Total Test Inference Time: {total_time:.2f} seconds")
print(f"Average Latency: {latency_per_sample:.2f} ms per bird call")

In [ ]:
# ── MACs & FLOPs ───────────────────────────────

from fvcore.nn import FlopCountAnalysis
import torch

# 1. Create a dummy input that matches HuBERT's expected wave shape (1 batch, 160,000 samples)
dummy_wave = torch.randn(1, 160000).to(model.device)

# 2. Run the Flop analyzer
print("Calculating HuBERT MACs...")
flops_hubert = FlopCountAnalysis(model, dummy_wave)

# 3. Suppress the massive layer-by-layer printout and just get the total
flops_hubert.unsupported_ops_warnings(False)

macs_total = flops_hubert.total()
flops_total = macs_total * 2

print("\n" + "="*40)
print(f"HuBERT Computational Complexity")
print("="*40)
print(f"Total MACs (GigaMACs): {macs_total / 1e9:.2f} GMACs")
print(f"Total FLOPs (GigaFLOPs): {flops_total / 1e9:.2f} GFLOPs")
print(f"Total Parameters:      {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

In [ ]:
# ── PLOT LOSS CURVES ────────────────────────────────────────────────────────
# extract the internal logs from trainer object
history = trainer_birdclef.state.log_history

train_loss = [x['loss'] for x in history if 'loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

epochs_train = range(1, len(train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_train, train_loss, label='Training Loss', color='blue')
plt.plot(np.linspace(1, len(train_loss), len(val_loss)), val_loss, label='Validation Loss', color='orange', marker='o')
plt.title('HuBERT Fine-Tuning: Loss Curve')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ── PLOT CONFUSION MATRIX ───────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

# Get the model's raw predictions on the test dataset
predictions_output = trainer_birdclef.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

# Use your unique labels to label the axes
# (Assuming you created 'unique_labels' earlier as a sorted list of your bird names)
unique_labels = sorted(list(set(labels)))

# Calculate the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Seaborn plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('HuBERT Bird Classification Confusion Matrix')
plt.xlabel('Predicted Bird Species')
plt.ylabel('Actual Bird Species')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHOW CLASS-LEVEL REPORT ───────────────────────────────────────────────────

from sklearn.metrics import classification_report

# 1. Get predictions
print("Generating class-level report...")
predictions_output = trainer_birdclef.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

# 2. Get the bird names in the correct order
# id2label should look like {0: "Barn swallow", 1: "Common raven", ...}
target_names = [id2label[i] for i in range(10)]

# 3. Print the holy grail of research tables
report = classification_report(y_true, y_pred, target_names=target_names)
print("\n" + "="*60)
print("CLASS-LEVEL PERFORMANCE REPORT")
print("="*60)
print(report)

In [ ]:
# Function to preprocess a single audio sample for testing
def load_audio_file(path, max_length=160000):
  audio, sr = librosa.load(path, sr=16000)

  # Truncate or pad the audio to match max_length
  if len(audio) > max_length:
      audio = audio[:max_length]
  elif len(audio) < max_length:
      audio = np.pad(audio, (0, max_length - len(audio)), mode='constant')

  return audio


# Function to translate label to bird name
def decode_label(label):
  species_labels = {
      "LABEL_0": "Barn swallow",
      "LABEL_1": "Common raven",
      "LABEL_2": "Curve billed thrasher",
      "LABEL_3": "European starling",
      "LABEL_4": "House wren",
      "LABEL_5": "Northern cardinal",
      "LABEL_6": "Red crossbill",
      "LABEL_7": "Red winged blackbird",
      "LABEL_8": "Song sparrow",
      "LABEL_9": "Spotted towhee"
  }
  return species_labels[label]


# Mapping to bird facts
bird_facts = {
    "Barn swallow": "They are the most widespread swallow in the world, but in America, they love building mud nests right on human porches.",
    "Common raven": "Ravens are crazy smart—they actually use logic to solve puzzles and have been known to sled down snowy roofs just for fun.",
    "Curve billed thrasher": "The jazz musicians of the desert; they don't just sing, they 'thash' their bills through debris to find insects while singing.",
    "European starling": "Total acoustics experts! They can mimic human speech, car alarms, and even the songs of other birds in your dataset.",
    "House wren": "Don't let the size fool you; these tiny guys are fiercely territorial and will aggressively dive-bomb much larger animals.",
    "Northern cardinal": "Unlike most songbirds, both males and females sing. They use their loud, clear whistles to communicate like a high-pitched walkie-talkie.",
    "Red crossbill": "Their beaks are literally crossed! This specialized 'tool' lets them pry open pine cones that other birds can't touch.",
    "Red winged blackbird": "The 'bouncers' of the marsh. Their 'Conk-la-ree!' song is so loud and distinct it's usually the first thing our model picks up in noisy data.",
    "Song sparrow": "Every male Song Sparrow has a unique, customized song that he uses to defend his territory—like a musical fingerprint.",
    "Spotted towhee": "They are the kings of the 'double-scratch'—a hop-and-kick move they use to find food in dead leaves, often making more noise than their song!"
}

In [ ]:
# ── Setup Demo ────────────────────────────

from transformers import pipeline
from IPython.display import display, Image, Audio
import time

# Setup the pipeline
classification_tool = pipeline(
    "audio-classification",
    model=model,
    feature_extractor=feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Demo / Inference
def bert_demo(file_path):
    # # Process the file you uploaded
    # input_audio = load_audio_file(file_path)

    # Use the pipeline to get the result
    probabilities = classification_tool(file_path)

    # probabilities look like -> (e.g., {'label': 'LABEL_1', 'score': 0.98})
    result = probabilities[0]
    predicted_label_id = result['label']
    confidence = result['score'] * 100
    bird_name = decode_label(predicted_label_id)

    fact = bird_facts.get(bird_name, "Truly one of America's finest birbs.")

    # --- Output ---
    print(f"[Bird 1]: Yo bert\n")
    time.sleep(1.5)
    print(f"[Bert]: Yo\n")
    time.sleep(1.5)
    print(f"[Bird 1]: What kinda bird iz dith?\n")
    time.sleep(1.5)
    print(f"[Bert]: I'm {confidence:.1f}% sure that's a {bird_name}:")
    time.sleep(1.5)
    print("-" * 40)

    # Display Image from Drive
    image_path = f'/content/drive/MyDrive/HuBERT/bird_photos/{bird_name}.jpg'
    # print(f"DEBUG: Looking for image at {image_path}")
    if os.path.exists(image_path):
        display(Image(filename=image_path, width=400))

    # Play first example of this bird from our test set
    # Standardize the name Bert predicted
    # This turns "Common raven" into "common_raven" to match the dataset
    search_name = bird_name.lower().replace(" ", "_")
    sample_found = False
    for i in range(len(test_dataset)):
        # Convert the numerical label in the dataset back to its string name
        # e.g., id2label[1] -> "common_raven"
        dataset_bird_name = id2label[test_dataset[i]['label']]

        # Standardize the dataset name too, just to be safe
        dataset_bird_name = dataset_bird_name.lower().replace(" ", "_")

        if dataset_bird_name == search_name:
            # Pull the data
            audio_data = test_dataset[i]['input_values']
            # Convert to a np array (this handles lists or tensors)
            audio_array = np.array(audio_data)
            # flatten it
            audio_array = audio_array.flatten()

            display(Audio(audio_array, rate=16000)) # display audio!
            sample_found = True
            break

    print(f"[Bert]: {fact}\n\n")
    time.sleep(2.5)
    print(f"[Bert]: here's your input back lil bro-")
    time.sleep(1)
    display(Audio(file_path, rate=16000))
    time.sleep(1.5)
    print(f"\n[\"your input back lil bro-\"]: Hi")


In [ ]:
# ── Load Saved HuBERT Model ────────────────────────────
# 1. run !pip install block

# 2. run import block

# 3. run Mount Drive and id2label blocks

# 4. run Load Dataset

# 5. run Demo setup

from transformers import pipeline, AutoModelForAudioClassification, AutoFeatureExtractor

# 1. Point to the saved model on Drive
saved_model_path = '/content/drive/MyDrive/HuBERT_Thai/bert_model_150_10_8_16' # <---- change if need
base_extractor_path = "facebook/hubert-base-ls960"

print("Loading saved model from Drive...")
# 2. Load the trained model and the original feature extractor
loaded_model = AutoModelForAudioClassification.from_pretrained(saved_model_path)
loaded_extractor = AutoFeatureExtractor.from_pretrained(base_extractor_path)

# 3. Rebuild the pipeline for Bert
classification_tool = pipeline(
    "audio-classification",
    model=loaded_model,
    feature_extractor=loaded_extractor,
    device=0 if torch.cuda.is_available() else -1
)

print("Bert is online and ready for inference!")

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import Trainer, TrainingArguments
from sklearn.metrics import confusion_matrix, classification_report
from fvcore.nn import FlopCountAnalysis

print("=" * 60)
print("1. HuBERT ARCHITECTURE & COMPLEXITY (LOADED MODEL)")
print("=" * 60)

# A. Calculate Parameters
num_params = sum(p.numel() for p in loaded_model.parameters())
print(f"Total Parameters: {num_params / 1e6:.2f} Million")

# B. Calculate MACs and FLOPs using a Wrapper
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x).logits

wrapped_model = ModelWrapper(loaded_model).eval()
# HuBERT expects a 1D raw waveform input: [Batch, Samples]
dummy_waveform = torch.randn(1, 160000).to(loaded_model.device)

flops_hubert = FlopCountAnalysis(wrapped_model, dummy_waveform)
flops_hubert.unsupported_ops_warnings(False)
macs_total = flops_hubert.total()

print(f"Total MACs:       {macs_total / 1e9:.2f} GMACs")
print(f"Total FLOPs:      {(macs_total * 2) / 1e9:.2f} GFLOPs")


print("\n" + "=" * 60)
print("2. HuBERT INFERENCE & ACCURACY METRICS")
print("=" * 60)

# C. Setup Trainer for Evaluation
hubert_eval_trainer = Trainer(
    model=loaded_model,
    args=TrainingArguments(output_dir="/tmp/hubert_eval", per_device_eval_batch_size=16, report_to="none"),
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# D. Run Prediction and Time it
start_inference = time.time()
hubert_loaded_results = hubert_eval_trainer.predict(test_dataset)
end_inference = time.time()

latency_per_sample = ((end_inference - start_inference) / len(test_dataset)) * 1000

print(f"  Accuracy   : {hubert_loaded_results.metrics['test_accuracy']:.4f}")
print(f"  F1 Score   : {hubert_loaded_results.metrics['test_f1']:.4f}")
print(f"  Precision  : {hubert_loaded_results.metrics['test_precision']:.4f}")
print(f"  Recall     : {hubert_loaded_results.metrics['test_recall']:.4f}")
print(f"  Latency    : {latency_per_sample:.2f} ms per waveform")

# E. Generate Reports and Graphs
y_pred = np.argmax(hubert_loaded_results.predictions, axis=1)
y_true = hubert_loaded_results.label_ids
unique_labels = sorted(list(set(labels)))
target_names = [id2label[i] for i in range(10)]

print("\n" + "=" * 60)
print("3. CLASS-LEVEL PERFORMANCE REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('HuBERT Classification Confusion Matrix (Loaded Model)')
plt.xlabel('Predicted Bird Species')
plt.ylabel('Actual Bird Species')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Play Demo
test_file = '/content/drive/MyDrive/HuBERT/test_audio/the-sound-of-swallows.mp3'
bert_demo(test_file)